# Preprocesamiento Corregido — Sistema CAD Retinopatía Diabética

**Versión corregida** del pipeline de preprocesamiento. Correcciones aplicadas respecto al notebook anterior:

- ✅ **BMI conservada**: se excluye explícitamente de la lista VIF; WEIGHT se elimina en su lugar.
- ✅ **Binarias excluidas del escalado**: `RobustScaler` solo se aplica a variables continuas.
- ✅ **Binarización post-imputación**: variables binarias se redondean después de KNN para eliminar decimales.
- ✅ **Chi-cuadrado añadido**: análisis estadístico completo para variables binarias.
- ✅ **Tres modelos comparados**: Random Forest, XGBoost y Regresión Logística.
- ✅ **Features de interacción escaladas**: se escalan en el mismo paso que las continuas.
- ✅ **Métricas orientadas a tamizaje**: optimización por F2-score + evaluación con umbral ajustado.

## 1. Importaciones

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib

from IPython.display import display
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    f1_score, fbeta_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve,
    classification_report
)
from sklearn.feature_selection import SelectFromModel
from statsmodels.stats.outliers_influence import variance_inflation_factor

from xgboost import XGBClassifier
import shap

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

RANDOM_STATE = 42
print('Todas las librerías importadas correctamente.')

## 2. Carga y exploración inicial del dataset

In [ ]:
df = pd.read_excel("/content/raw data (1).xlsx")

print(f"Registros: {df.shape[0]}   |   Columnas: {df.shape[1]}")
print(f"\nDistribución de la variable objetivo:")
print(df['Disease Type'].value_counts())
print(f"Balance: {df['Disease Type'].value_counts(normalize=True).round(3).to_dict()}")
df.head()

In [ ]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols     = df.select_dtypes(include=['number']).columns.tolist()

print(f"Variables categóricas ({len(categorical_cols)}): {categorical_cols}")
print(f"\nVariables numéricas ({len(numeric_cols)}): {numeric_cols}")
df.describe()

## 3. Limpieza inicial

### 3.1 Eliminación de variables con >80 % de valores faltantes

In [ ]:
missing_percent = df.isnull().mean().sort_values(ascending=False)
high_missing_cols = missing_percent[missing_percent > 0.8]

print(f"Columnas con >80% de faltantes ({len(high_missing_cols)}):")
print(high_missing_cols.to_string())

cleaned_df = df.drop(columns=high_missing_cols.index)
cleaned_df = cleaned_df.drop(columns=['Case_ID'], errors='ignore')

print(f"\nDimensiones tras limpieza: {cleaned_df.shape}")

In [ ]:
remaining_missing = cleaned_df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)

print("Variables con valores faltantes restantes:")
display(remaining_missing.to_frame(name='Valores Faltantes').head(30))

## 4. División del dataset (80/20) — ANTES de cualquier transformación

> **Criterio metodológico**: el split se realiza primero para evitar que la información del conjunto de prueba
> contamine la imputación, el escalado o la selección de variables (data leakage).

In [ ]:
X = cleaned_df.drop('Disease Type', axis=1)
y = cleaned_df['Disease Type']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"Distribución y_train: {y_train.value_counts().to_dict()}")
print(f"Distribución y_test:  {y_test.value_counts().to_dict()}")

## 5. Identificación de variables binarias

> **Corrección aplicada**: las variables binarias se identifican ANTES del escalado para excluirlas
> correctamente del `RobustScaler`. Variables con exactamente 2 valores únicos son tratadas como binarias.

In [ ]:
# Identificar variables binarias basadas en X completo (antes del split)
binary_vars = [col for col in X.columns if X[col].nunique() == 2]
continuous_vars = [col for col in X.columns if col not in binary_vars]

print(f"Variables binarias identificadas ({len(binary_vars)}):")
print(binary_vars)
print(f"\nVariables continuas ({len(continuous_vars)}): {len(continuous_vars)} columnas")

## 6. Imputación con KNNImputer

> **Criterio**: `fit_transform` exclusivamente en `X_train`; `transform` en `X_test` para simular datos nuevos.
> Después de imputar, las variables binarias se redondean para eliminar valores decimales intermedios.

In [ ]:
cols_with_missing = X_train.columns[X_train.isnull().any()].tolist()

if cols_with_missing:
    imputer = KNNImputer(n_neighbors=5)

    # Ajuste solo en train
    X_train_imp = imputer.fit_transform(X_train[cols_with_missing])
    X_train[cols_with_missing] = pd.DataFrame(
        X_train_imp, columns=cols_with_missing, index=X_train.index
    )

    # Solo transformación en test
    X_test_imp = imputer.transform(X_test[cols_with_missing])
    X_test[cols_with_missing] = pd.DataFrame(
        X_test_imp, columns=cols_with_missing, index=X_test.index
    )

    joblib.dump(imputer, 'knn_imputer.joblib')
    print(f"KNNImputer ajustado y guardado.")

    # ─── CORRECCIÓN: binarizar variables binarias post-imputación ───
    binary_imputed = [c for c in cols_with_missing if c in binary_vars]
    if binary_imputed:
        X_train[binary_imputed] = X_train[binary_imputed].round().astype(int)
        X_test[binary_imputed]  = X_test[binary_imputed].round().astype(int)
        print(f"Variables binarias re-binarizadas post-imputación: {binary_imputed}")

    print(f"Faltantes en X_train tras imputación: {X_train.isnull().sum().sum()}")
    print(f"Faltantes en X_test  tras imputación: {X_test.isnull().sum().sum()}")
else:
    print("No hay valores faltantes en X_train.")

## 7. Escalado con RobustScaler — solo variables continuas

> **Corrección aplicada**: se excluyen explícitamente las variables binarias del escalado.
> Los modelos de árbol son invariantes al escalado, pero esta corrección es esencial para
> la Regresión Logística que se entrenará más adelante.

In [ ]:
# Variables continuas presentes en el dataset tras limpieza inicial
continuous_cols_present = [
    c for c in X_train.columns
    if c in continuous_vars and c in X_train.columns
]

scaler = RobustScaler()

X_train[continuous_cols_present] = scaler.fit_transform(X_train[continuous_cols_present])
X_test[continuous_cols_present]  = scaler.transform(X_test[continuous_cols_present])

joblib.dump(scaler, 'robust_scaler.joblib')
print(f"RobustScaler aplicado a {len(continuous_cols_present)} variables continuas.")
print(f"Variables binarias conservadas sin escalar: {[c for c in binary_vars if c in X_train.columns]}")
display(X_train[continuous_cols_present].head(3))

## 8. Tratamiento de colinealidad (VIF)

> **Corrección crítica**: BMI se conserva explícitamente. Es un predictor clínicamente relevante
> para Retinopatía Diabética según las Guías ADA 2026. WEIGHT se elimina en su lugar, ya que
> BMI y WEIGHT son colineales por definición matemática pero BMI tiene mayor valor clínico.
>
> Solo se calcula VIF sobre variables continuas (las binarias no participan en el cálculo de VIF).

In [ ]:
import statsmodels.api as sm

# VIF solo sobre continuas, sin NaN
vif_df = X_train[continuous_cols_present].dropna()
vif_data = pd.DataFrame()
vif_data['Variable'] = vif_df.columns
vif_data['VIF'] = [
    variance_inflation_factor(vif_df.values, i)
    for i in range(vif_df.shape[1])
]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("VIF calculado (variables continuas):")
display(vif_data)

In [ ]:
# ─── CORRECCIÓN: lista VIF con BMI explícitamente protegida ───
# Se eliminan variables con VIF > 10 del notebook original,
# pero se garantiza que BMI NO esté en la lista de eliminación.

# Variables con VIF > 10 identificadas en el análisis previo:
high_vif_cols = [
    'WEIGHT',   # Colineal con BMI → se elimina WEIGHT, no BMI
    'TC',
    'LDL_C',
    'HB',
    'PCV',
    'TBILI',
    'DBILI',
    'TP',
    'ALB',
    'GLO'
]

# Protección explícita: BMI no debe estar en la lista
assert 'BMI' not in high_vif_cols, "ERROR: BMI está en la lista VIF. Corrígela antes de continuar."
print("✓ BMI confirmada como NO eliminada por VIF.")

X_train = X_train.drop(columns=high_vif_cols, errors='ignore')
X_test  = X_test.drop(columns=high_vif_cols, errors='ignore')

# Actualizar la lista de continuas tras VIF
continuous_cols_present = [c for c in continuous_cols_present if c not in high_vif_cols]

print(f"Columnas eliminadas por VIF: {high_vif_cols}")
print(f"Dimensiones X_train tras VIF: {X_train.shape}")
print(f"BMI presente en X_train: {'BMI' in X_train.columns}")

## 9. Análisis estadístico bivariado

### 9.1 Variables continuas: t-test + Mann-Whitney U (solo en train)

In [ ]:
numeric_for_stats = [
    col for col in X_train.select_dtypes(include=['float64', 'int64']).columns
    if col not in binary_vars
]

stat_results = []
for col in numeric_for_stats:
    if col not in X_train.columns:
        continue
    g0 = X_train.loc[y_train == 0, col].dropna()
    g1 = X_train.loc[y_train == 1, col].dropna()

    try:
        _, t_p = ttest_ind(g0, g1, equal_var=False)
    except Exception:
        t_p = np.nan
    try:
        _, u_p = mannwhitneyu(g0, g1, alternative='two-sided')
    except Exception:
        u_p = np.nan

    stat_results.append({
        'Variable': col,
        'Media_SinRD': round(g0.mean(), 3),
        'Media_ConRD': round(g1.mean(), 3),
        'p_ttest': round(t_p, 4) if not np.isnan(t_p) else np.nan,
        'p_mannwhitney': round(u_p, 4) if not np.isnan(u_p) else np.nan,
        'Significativo': 'Sí' if (not np.isnan(u_p) and u_p < 0.05) else 'No'
    })

stats_df = pd.DataFrame(stat_results).sort_values('p_mannwhitney')
print("Top 20 variables continuas más significativas (Mann-Whitney U):")
display(stats_df.head(20))

### 9.2 Variables binarias: Chi-cuadrado

> **Corrección aplicada**: el notebook anterior no realizaba análisis estadístico para variables binarias.
> Se añade chi-cuadrado para todas las variables con exactamente 2 valores únicos.

In [ ]:
binary_in_train = [col for col in binary_vars if col in X_train.columns]

chi2_results = []
for col in binary_in_train:
    contingency = pd.crosstab(X_train[col], y_train)
    try:
        chi2, p, dof, expected = chi2_contingency(contingency)
        # Prevalencia de RD en cada grupo
        prop = X_train.groupby(col)[col].count()   # conteo por grupo
        rd_by_group = pd.crosstab(X_train[col], y_train, normalize='index') * 100
        pct_0 = round(rd_by_group.loc[0, 1], 1) if (0 in rd_by_group.index and 1 in rd_by_group.columns) else np.nan
        pct_1 = round(rd_by_group.loc[1, 1], 1) if (1 in rd_by_group.index and 1 in rd_by_group.columns) else np.nan
    except Exception:
        chi2, p, pct_0, pct_1 = np.nan, np.nan, np.nan, np.nan

    chi2_results.append({
        'Variable': col,
        '%RD_si_ausente': pct_0,
        '%RD_si_presente': pct_1,
        'Chi2': round(chi2, 3) if not np.isnan(chi2) else np.nan,
        'p_valor': round(p, 4) if not np.isnan(p) else np.nan,
        'Significativo': 'Sí' if (not np.isnan(p) and p < 0.05) else 'No'
    })

chi2_df = pd.DataFrame(chi2_results).sort_values('p_valor')
print("Análisis chi-cuadrado — Variables binarias:")
display(chi2_df)

## 10. Selección de variables

### 10.1 Permutation Importance (Random Forest sobre dataset completo post-VIF)

In [ ]:
from sklearn.inspection import permutation_importance

rf_selector = RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
rf_selector.fit(X_train, y_train)

perm_result = permutation_importance(
    rf_selector, X_test, y_test,
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1, scoring='roc_auc'
)

perm_df = pd.DataFrame({
    'Variable': X_test.columns,
    'Importance_Mean': perm_result.importances_mean,
    'Importance_Std':  perm_result.importances_std
}).sort_values('Importance_Mean', ascending=False).reset_index(drop=True)

print("Top 20 variables por Permutation Importance (AUC):")
display(perm_df.head(20))

In [ ]:
top15 = perm_df.head(15)

fig, ax = plt.subplots(figsize=(11, 7))
sns.barplot(
    x='Importance_Mean', y='Variable', data=top15,
    palette='viridis', hue='Variable', legend=False, ax=ax
)
ax.set_title('Permutation Importance — Top 15 variables (AUC)', fontsize=15)
ax.set_xlabel('Importancia media (AUC)', fontsize=13)
ax.set_ylabel('Variable', fontsize=13)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

### 10.2 Definición del conjunto de variables seleccionadas

Se combinan los criterios de Permutation Importance, significancia estadística (Mann-Whitney / Chi-cuadrado)
y criterio clínico del documento médico (HbA1c y microalbuminuria como marcadores centrales de RD).
BMI se incluye explícitamente según la corrección metodológica aplicada.

In [ ]:
# Variables seleccionadas por criterio estadístico + clínico + permutation importance
# Verificar cuáles están disponibles en X_train (pueden variar según el dataset cargado)

candidate_features = [
    # ── Marcadores glucémicos (base clínica según ADA 2026 y documento médico) ──
    'HBA1C',          # Hemoglobina glicosilada: predictor principal RD
    'GLU',            # Glucosa en ayunas
    'GSP',            # Proteína sérica glucosilada

    # ── Marcadores renales / microvasculares ──
    'ALB_CR',         # Relación albúmina-creatinina: microalbuminuria, daño endotelial
    'UPR_24',         # Microalbuminuria 24h: predictor más importante en artículo ref.
    'SCR',            # Creatinina sérica
    'BU',             # Urea en sangre

    # ── Variables hemodinámicas ──
    'Systolic blood pressure',
    'Diastolic blood pressure',

    # ── Variables antropométricas ──
    'BMI',            # CONSERVADA explícitamente (corrección VIF)
    'AGE',

    # ── Marcadores inflamatorios / hepáticos ──
    'CRP',            # Proteína C reactiva
    'GGT',

    # ── Variables binarias con significancia estadística (chi-cuadrado) ──
    'NEPHROPATHY',    # Nefropatía: fuertemente asociada a RD
    'HYPERTENTION',   # Hipertensión: factor de riesgo en guías ADA
    'HYPERLIPIDEMIA',
    'gender',
]

# Filtrar solo las que existen en el dataset
selected_features = [f for f in candidate_features if f in X_train.columns]

print(f"Variables seleccionadas disponibles ({len(selected_features)}):")
for i, f in enumerate(selected_features, 1):
    marker = '🔬' if f in binary_vars else '📊'
    print(f"  {i:2d}. {marker} {f}")

## 11. Feature engineering — interacciones clínicas

> **Corrección aplicada**: las features de interacción se crean ANTES del paso de selección final
> para que puedan ser escaladas junto con las demás variables continuas si fuese necesario.
>
> Las tres interacciones capturan mecanismos fisiopatológicos documentados:
> - **Riesgo_Acumulado**: exposición acumulada a hiperglucemia crónica (HBA1C × AGE)
> - **Indice_Dano_Microvascular**: co-daño renal y glucémico (ALB_CR × HBA1C)
> - **Ratio_Glucosa_Estabilidad**: desproporción entre glucosa puntual y control crónico (GLU / HBA1C)

In [ ]:
def create_interaction_features(df_in, hba1c_col='HBA1C', age_col='AGE',
                                alb_cr_col='ALB_CR', glu_col='GLU'):
    """Crea features de interacción clínica. Maneja columnas ausentes gracefully."""
    df_out = df_in.copy()

    if hba1c_col in df_out.columns and age_col in df_out.columns:
        df_out['Riesgo_Acumulado'] = df_out[hba1c_col] * df_out[age_col]

    if alb_cr_col in df_out.columns and hba1c_col in df_out.columns:
        df_out['Indice_Dano_Microvascular'] = df_out[alb_cr_col] * df_out[hba1c_col]

    if glu_col in df_out.columns and hba1c_col in df_out.columns:
        df_out['Ratio_Glucosa_Estabilidad'] = np.where(
            df_out[hba1c_col] == 0, 0,
            df_out[glu_col] / df_out[hba1c_col]
        )

    return df_out


X_train = create_interaction_features(X_train)
X_test  = create_interaction_features(X_test)

interaction_features = ['Riesgo_Acumulado', 'Indice_Dano_Microvascular', 'Ratio_Glucosa_Estabilidad']
interaction_features_present = [f for f in interaction_features if f in X_train.columns]

# ─── Escalar las features de interacción (son continuas) ───
if interaction_features_present:
    scaler_int = RobustScaler()
    X_train[interaction_features_present] = scaler_int.fit_transform(X_train[interaction_features_present])
    X_test[interaction_features_present]  = scaler_int.transform(X_test[interaction_features_present])
    joblib.dump(scaler_int, 'robust_scaler_interactions.joblib')
    print(f"Features de interacción creadas y escaladas: {interaction_features_present}")

# Añadir al conjunto de features seleccionadas
selected_features += interaction_features_present
selected_features = [f for f in selected_features if f in X_train.columns]  # Re-filtrar

print(f"\nTotal features finales para modelado: {len(selected_features)}")
print(selected_features)

In [ ]:
X_train_sel = X_train[selected_features].copy()
X_test_sel  = X_test[selected_features].copy()

print(f"X_train_sel: {X_train_sel.shape}")
print(f"X_test_sel:  {X_test_sel.shape}")
print(f"\nFaltantes en X_train_sel: {X_train_sel.isnull().sum().sum()}")
print(f"Faltantes en X_test_sel:  {X_test_sel.isnull().sum().sum()}")

## 12. Visualizaciones exploratorias

In [ ]:
# Boxplots de variables continuas clave vs. variable objetivo
key_continuous = [f for f in ['HBA1C', 'ALB_CR', 'BMI', 'AGE', 'CRP', 'SCR']
                  if f in cleaned_df.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(key_continuous):
    sns.boxplot(
        x='Disease Type', y=col, data=cleaned_df,
        palette='viridis', hue='Disease Type', legend=False, ax=axes[i]
    )
    axes[i].set_title(f'Distribución de {col}', fontsize=13)
    axes[i].set_xlabel('Estado RD', fontsize=11)
    axes[i].set_ylabel(col, fontsize=11)
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(['Sin RD', 'Con RD'])

plt.suptitle('Distribución de variables clínicas clave según presencia de RD', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Proporción de RD para variables binarias significativas
key_binary = [f for f in ['NEPHROPATHY', 'HYPERLIPIDEMIA', 'HYPERTENTION', 'gender']
              if f in cleaned_df.columns]

n_cols = min(len(key_binary), 4)
fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 6))
if n_cols == 1:
    axes = [axes]

for i, var in enumerate(key_binary):
    prop_dr = cleaned_df.groupby(var)['Disease Type'].mean().reset_index()
    prop_dr['Disease Type'] = prop_dr['Disease Type'] * 100

    sns.barplot(
        x=var, y='Disease Type', data=prop_dr,
        ax=axes[i], palette='viridis', hue=var, legend=False
    )
    axes[i].set_title(f'% RD por {var}', fontsize=13)
    axes[i].set_xlabel(f'{var}', fontsize=11)
    axes[i].set_ylabel('% con Retinopatía', fontsize=11)
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels([f'Sin {var}', f'Con {var}'])
    for container in axes[i].containers:
        axes[i].bar_label(container, fmt='%.1f%%')

plt.suptitle('Prevalencia de RD según variables binarias clave', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de correlación de Spearman entre variables seleccionadas continuas
corr_vars = [f for f in ['HBA1C', 'GLU', 'ALB_CR', 'BMI',
                          'Systolic blood pressure', 'AGE', 'SCR', 'CRP', 'GGT']
             if f in cleaned_df.columns]

corr_matrix = cleaned_df[corr_vars].corr(method='spearman')

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr_matrix, annot=True, cmap='coolwarm', fmt='.2f',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Correlación de Spearman'}
)
ax.set_title('Mapa de calor — Correlación de Spearman (variables continuas clave)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 13. Entrenamiento y optimización de modelos

Se entrenan **tres modelos** con `StratifiedKFold` (5 folds) y `RandomizedSearchCV` optimizando el **F2-score**
(prioriza recall sobre precisión, apropiado para tamizaje clínico):

1. **Regresión Logística** — interpretable, genera Odds Ratios clínicos
2. **Random Forest** — robusto a outliers y no linealidades
3. **XGBoost** — mayor capacidad predictiva

> El F2-score es `Fβ` con `β=2`, lo que pondera el recall dos veces más que la precisión.

In [ ]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import RandomizedSearchCV

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
f2_scorer = make_scorer(fbeta_score, beta=2)

print("Configuración de validación cruzada:")
print(f"  Estrategia: StratifiedKFold (n_splits=5)")
print(f"  Métrica de optimización: F2-score (prioriza recall)")
print(f"  Dataset de entrenamiento: {X_train_sel.shape}")

### 13.1 Regresión Logística

In [ ]:
lr_param_dist = {
    'C':          [0.001, 0.01, 0.1, 0.5, 1, 5, 10],
    'penalty':    ['l1', 'l2'],
    'solver':     ['liblinear', 'saga'],
    'class_weight': [None, 'balanced'],
}

lr_base = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

lr_search = RandomizedSearchCV(
    estimator=lr_base,
    param_distributions=lr_param_dist,
    n_iter=30,
    scoring=f2_scorer,
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)
lr_search.fit(X_train_sel, y_train)

best_lr = lr_search.best_estimator_
print(f"\nMejores parámetros LR: {lr_search.best_params_}")
print(f"Mejor F2-score CV:     {lr_search.best_score_:.4f}")

### 13.2 Random Forest

In [ ]:
rf_param_dist = {
    'n_estimators':    [100, 200, 300],
    'max_depth':       [10, 20, None],
    'min_samples_leaf':[1, 2, 4],
    'min_samples_split':[2, 5, 10],
    'max_features':    ['sqrt', 'log2'],
    'class_weight':    [None, 'balanced'],
}

rf_base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_dist,
    n_iter=40,
    scoring=f2_scorer,
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)
rf_search.fit(X_train_sel, y_train)

best_rf = rf_search.best_estimator_
print(f"\nMejores parámetros RF: {rf_search.best_params_}")
print(f"Mejor F2-score CV:     {rf_search.best_score_:.4f}")

### 13.3 XGBoost

In [ ]:
xgb_param_dist = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [3, 5, 7],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'scale_pos_weight': [1.0, 1.5, 2.0],  # Penalizar falsos negativos
    'reg_alpha':        [0, 0.1, 0.5],
    'reg_lambda':       [1, 1.5, 2],
}

xgb_base = XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    n_jobs=-1
)

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=40,
    scoring=f2_scorer,
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)
xgb_search.fit(X_train_sel, y_train)

best_xgb = xgb_search.best_estimator_
print(f"\nMejores parámetros XGBoost: {xgb_search.best_params_}")
print(f"Mejor F2-score CV:          {xgb_search.best_score_:.4f}")

## 14. Evaluación completa de los tres modelos

Se evalúan con **umbral estándar (0.50)** y **umbral de tamizaje (0.35)** para
maximizar recall a costa de mayor tasa de falsos positivos.

In [ ]:
def evaluate_model(model, X_test, y_test, threshold=0.5, model_name='Modelo'):
    """
    Evalúa un clasificador binario con umbral personalizable.
    Retorna un diccionario de métricas y muestra la matriz de confusión.
    """
    y_prob  = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_prob >= threshold).astype(int)

    acc   = accuracy_score(y_test, y_pred)
    prec  = precision_score(y_test, y_pred, zero_division=0)
    rec   = recall_score(y_test, y_pred, zero_division=0)
    f1    = f1_score(y_test, y_pred, zero_division=0)
    f2    = fbeta_score(y_test, y_pred, beta=2, zero_division=0)
    auc   = roc_auc_score(y_test, y_prob)
    cm    = confusion_matrix(y_test, y_pred)

    tn, fp, fn, tp = cm.ravel()
    spec  = tn / (tn + fp) if (tn + fp) > 0 else 0

    print(f"\n{'='*60}")
    print(f"  {model_name}  (umbral = {threshold})")
    print(f"{'='*60}")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  Precision   : {prec:.4f}")
    print(f"  Recall      : {rec:.4f}")
    print(f"  Specificity : {spec:.4f}")
    print(f"  F1-score    : {f1:.4f}")
    print(f"  F2-score    : {f2:.4f}")
    print(f"  AUC-ROC     : {auc:.4f}")
    print(f"  FN (falsos negativos): {fn}   |   FP (falsos positivos): {fp}")

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Sin RD', 'Con RD']).plot(
        cmap='Blues', ax=ax
    )
    ax.set_title(f'Matriz de Confusión: {model_name}\n(umbral={threshold})', fontsize=12)
    plt.tight_layout()
    plt.show()

    return {
        'Modelo': model_name, 'Umbral': threshold,
        'Accuracy': round(acc, 4), 'Precision': round(prec, 4),
        'Recall': round(rec, 4), 'Specificity': round(spec, 4),
        'F1': round(f1, 4), 'F2': round(f2, 4),
        'AUC': round(auc, 4), 'FN': fn, 'FP': fp
    }

print("Función de evaluación definida.")

In [ ]:
all_results = []

# ── Umbral estándar 0.50 ──
for model, name in [
    (best_lr,  'Regresión Logística'),
    (best_rf,  'Random Forest'),
    (best_xgb, 'XGBoost')
]:
    res = evaluate_model(model, X_test_sel, y_test, threshold=0.50, model_name=name)
    all_results.append(res)

# ── Umbral de tamizaje 0.35 ──
for model, name in [
    (best_lr,  'Regresión Logística (tamizaje)'),
    (best_rf,  'Random Forest (tamizaje)'),
    (best_xgb, 'XGBoost (tamizaje)')
]:
    res = evaluate_model(model, X_test_sel, y_test, threshold=0.35, model_name=name)
    all_results.append(res)

## 15. Tabla comparativa consolidada de resultados

In [ ]:
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(['Umbral', 'F2'], ascending=[True, False])

print("Tabla comparativa completa de los tres modelos:")
display(results_df.to_string(index=False))

In [ ]:
# Gráfico de comparación de métricas principales (umbral 0.35 — orientado a tamizaje)
results_035 = results_df[results_df['Umbral'] == 0.35].copy()
metrics     = ['Accuracy', 'Recall', 'Precision', 'Specificity', 'F2', 'AUC']
models_035  = results_035['Modelo'].str.replace(' (tamizaje)', '', regex=False).tolist()

x = np.arange(len(metrics))
width = 0.25
colors = ['#2196F3', '#4CAF50', '#FF9800']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (_, row) in enumerate(results_035.iterrows()):
    vals = [row[m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=models_035[i], color=colors[i], alpha=0.85)
    ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=2)

ax.set_xlabel('Métrica', fontsize=13)
ax.set_ylabel('Valor', fontsize=13)
ax.set_title('Comparación de métricas — Tres modelos (umbral 0.35)', fontsize=14)
ax.set_xticks(x + width)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=11)
ax.axhline(0.9, color='red', linestyle='--', linewidth=0.8, alpha=0.6, label='Referencia 0.9')
plt.tight_layout()
plt.show()

## 16. Curvas ROC comparativas

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

colors_roc = ['#2196F3', '#4CAF50', '#FF9800']
for (model, name), color in zip(
    [(best_lr, 'Regresión Logística'), (best_rf, 'Random Forest'), (best_xgb, 'XGBoost')],
    colors_roc
):
    y_prob = model.predict_proba(X_test_sel)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.3f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Clasificador aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos (1 - Especificidad)', fontsize=13)
ax.set_ylabel('Tasa de Verdaderos Positivos (Sensibilidad)', fontsize=13)
ax.set_title('Curvas ROC — Comparación de los tres modelos', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 17. Interpretabilidad — SHAP

### 17.1 SHAP para Random Forest

In [ ]:
explainer_rf = shap.TreeExplainer(best_rf)
subset_test  = X_test_sel.sample(min(200, len(X_test_sel)), random_state=RANDOM_STATE)
shap_vals_rf = explainer_rf.shap_values(subset_test)

# Para sklearn RF, shap_values devuelve [clase0, clase1] o array 3D
if isinstance(shap_vals_rf, list):
    sv = shap_vals_rf[1]
elif shap_vals_rf.ndim == 3:
    sv = shap_vals_rf[:, :, 1]
else:
    sv = shap_vals_rf

shap.summary_plot(sv, subset_test, show=False)
plt.title('SHAP — Random Forest (Clase 1: Con RD)', fontsize=13)
plt.tight_layout()
plt.show()

### 17.2 SHAP para XGBoost

In [ ]:
explainer_xgb = shap.TreeExplainer(best_xgb)
shap_vals_xgb = explainer_xgb.shap_values(subset_test)

shap.summary_plot(shap_vals_xgb, subset_test, show=False)
plt.title('SHAP — XGBoost (Clase 1: Con RD)', fontsize=13)
plt.tight_layout()
plt.show()

### 17.3 Coeficientes de la Regresión Logística (Odds Ratios)

In [ ]:
coef_df = pd.DataFrame({
    'Variable': selected_features,
    'Coeficiente': best_lr.coef_[0],
    'Odds_Ratio': np.exp(best_lr.coef_[0])
}).sort_values('Odds_Ratio', ascending=False)

print("Coeficientes e Odds Ratios — Regresión Logística:")
display(coef_df)

fig, ax = plt.subplots(figsize=(10, max(6, len(selected_features) * 0.45)))
colors_coef = ['#E53935' if v > 0 else '#1E88E5' for v in coef_df['Coeficiente'].values]
ax.barh(coef_df['Variable'], coef_df['Coeficiente'], color=colors_coef, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente (escala log-odds)', fontsize=12)
ax.set_title('Coeficientes Regresión Logística\n(positivo = mayor riesgo de RD)', fontsize=13)
plt.tight_layout()
plt.show()

## 18. Análisis del trade-off de umbral (curva Precisión-Recall)

In [ ]:
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (model, name) in zip(
    axes,
    [(best_lr, 'Regresión Logística'), (best_rf, 'Random Forest'), (best_xgb, 'XGBoost')]
):
    y_prob   = model.predict_proba(X_test_sel)[:, 1]
    prec_arr, rec_arr, thresholds_arr = precision_recall_curve(y_test, y_prob)

    ax.plot(thresholds_arr, prec_arr[:-1], label='Precisión', color='#1565C0', linewidth=2)
    ax.plot(thresholds_arr, rec_arr[:-1],  label='Recall',    color='#C62828', linewidth=2)

    # Marcar umbrales de referencia
    for thr, col, lbl in [(0.50, 'gray', '0.50'), (0.35, 'orange', '0.35')]:
        ax.axvline(thr, color=col, linestyle='--', linewidth=1.2, alpha=0.7)
        ax.text(thr + 0.01, 0.05, lbl, color=col, fontsize=9)

    ax.set_xlabel('Umbral de decisión', fontsize=11)
    ax.set_ylabel('Valor', fontsize=11)
    ax.set_title(f'{name}\nTrade-off Precisión-Recall', fontsize=12)
    ax.legend(fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)

plt.suptitle('Análisis de umbral — Precisión vs. Recall', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 19. Resumen final y conclusiones del pipeline corregido

In [ ]:
print("=" * 70)
print("RESUMEN DEL PIPELINE CORREGIDO")
print("=" * 70)

print("\n[CORRECCIONES APLICADAS]")
print("  ✅ BMI conservada (protegida de eliminación por VIF)")
print("  ✅ Variables binarias excluidas del RobustScaler")
print("  ✅ Re-binarización de variables binarias post-KNN Imputer")
print("  ✅ Chi-cuadrado aplicado a variables binarias")
print("  ✅ Features de interacción escaladas antes de modelado")
print("  ✅ Tres modelos entrenados: LR, RF, XGBoost")
print("  ✅ Optimización por F2-score (prioriza recall)")

print("\n[VARIABLES FINALES DEL MODELO]")
for f in selected_features:
    tipo = 'Binaria' if f in binary_vars else 'Continua'
    print(f"  - {f:<40} ({tipo})")

print("\n[TABLA DE RESULTADOS — UMBRAL 0.35 (TAMIZAJE)]")
cols_show = ['Modelo', 'Accuracy', 'Recall', 'Precision', 'Specificity', 'F2', 'AUC', 'FN', 'FP']
results_035_clean = results_035[cols_show].copy()
results_035_clean['Modelo'] = results_035_clean['Modelo'].str.replace(' (tamizaje)', '', regex=False)
print(results_035_clean.to_string(index=False))

print("\n[TABLA DE RESULTADOS — UMBRAL 0.50 (ESTÁNDAR)]")
results_050 = results_df[results_df['Umbral'] == 0.50][cols_show]
print(results_050.to_string(index=False))

print("\n" + "=" * 70)
print("Pipeline completado correctamente.")
print("=" * 70)